# LLM Output Parsing

This notebook's purpose is to:
1. Import LLM output JSON
2. Parse into tuples for each entity for evaluation against ground truth

In [1]:
import os

import json

### Load LLM output JSON

In [2]:
with open("../data/results/biored_sample_extractions.json") as f:
    extractions = json.load(f)

### Parse Into Two Sets of Tuples
* `predictions_entities`
* `predictions_relationships`

**Includes PMID for duplicate handling as sets automatically remove duplicates**
* PMID avoids duplicates across papers

In [ ]:
predictions_entities = {
    (extraction["pmid"], e["name"], e["type"])
    for extraction in extractions
    for e in extraction["entities"]
}

predictions_relationships = {
    (extraction["pmid"], r["source"], r["relation"], r["target"])
    for extraction in extractions
    for r in extraction["relationships"]
}

# BioRED GT Parsing

### Load and Parse to Sets of Tuples

In [12]:
import pandas as pd

df = pd.read_csv("../data/processed/biored/br_dev_entity_relations.csv")

ground_truth_entities = (
    set(zip(df["pmid"], df["entity_1"], df["entity_1_type"])) |
    set(zip(df["pmid"], df["entity_2"], df["entity_2_type"]))
)

ground_truth_relationships = set(zip(df["pmid"], df["entity_1"], df["relation"], df["entity_2"]))

# Compute Quantitative Evaluation Metrics

In [15]:
def compute_prf(predictions, ground_truth):
    tp = len(predictions & ground_truth)
    fp = len(predictions - ground_truth)
    fn = len(ground_truth - predictions)

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

In [16]:
entity_metrics = compute_prf(predictions_entities, ground_truth_entities)
relationship_metrics = compute_prf(predictions_relationships, ground_truth_relationships)

print("Entities:", entity_metrics)
print("Relationships:", relationship_metrics)

Entities: {'precision': 0.0, 'recall': 0.0, 'f1': 0, 'tp': 0, 'fp': 758, 'fn': 850}
Relationships: {'precision': 0.0, 'recall': 0.0, 'f1': 0, 'tp': 0, 'fp': 831, 'fn': 1107}
